In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! pip install fastavro

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime
import fastavro
import logging
import uuid
import time

Fuente Archivos
* Locales
* En línea

In [ ]:
import os

# Permite que el usuario defina la ruta, o usa un valor por defecto
ruta_base = os.getenv("RUTA_DATOS", "./datos/")

ruta_credencial = f"{ruta_base}credenciales/"
ruta_log = f"{ruta_base}log/"
archivo_log = f"{ruta_log}logs_adquisicion_archivos.log"

# Crear carpetas si no existes
os.makedirs(ruta_credencial, exist_ok=True)
os.makedirs(ruta_log, exist_ok=True)

In [ ]:
# forzar para que acepte un df y una cadena
def print_profile(df: pd.DataFrame, name: str):
    print(f"\n=== {name} ===")
    print("shape:", df.shape) #dupla
    print("columnas:", list(df.columns))
    print("head:", df.info())
    display(df.head(5))

In [ ]:
# Configuración del logger
logger = logging.getLogger("pipeline_logger")
logger.setLevel(logging.INFO)

# Eliminar handlers previos si existen
if logger.hasHandlers():
    logger.handlers.clear()

file_handler = logging.FileHandler(archivo_log)
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler.setFormatter(formatter)

logger.addHandler(file_handler)

print("Logger configurado correctamente")

In [ ]:
#Variables de ejecución
run_id = str(uuid.uuid4())
inicio_proceso = time.time()
estado_global = "exitoso"
registros_extraidos = 0
registros_cargados = 0

logger.info(f"Inicio de pipeline | run_id={run_id}")

In [ ]:
# csv local
ruta_csv = f"{ruta_base}add/csv/alumnos_fuente_001.csv"

try:
  df_csv = pd.read_csv(ruta_csv) # contiene parametros por si me voy a eliminar ciertos registros
  print_profile(df_csv, "df_csv")
except Exception as e:
  print(f"Error al leer el archivo CSV: {e}")
  estado_global = "error"

In [ ]:
# ruta_csv = f"enlace_contenido"

try:
  df_csv_url = pd.read_csv(ruta_csv, sep=",")
  print_profile(df_csv_url, "df_csv_url")
except Exception as e:
  print(f"Error al leer el archivo CSV: {e}")
  estado_global = "error"

In [ ]:
# CSV desde un directorio
ruta_csv= f"{ruta_base}add/csv/"
try:
  archivos = [
      f for f in os.listdir(ruta_csv) if f.endswith(".csv")
  ]
  print(f"Numero de archivos", len(archivos))
  print(f"Archivos encontrados: {archivos}")
  dfs = []
  for a in archivos:
    df = pd.read_csv(f"{ruta_csv}{a}", sep=",")
    df["archivo_origen"] = a
    dfs.append(df)

  df.csv_dir = pd.concat(dfs, ignore_index=True)
  print_profile(df.csv_dir, "df.csv_dir")
except Exception as e:
  print(f"Error al leer el archivo CSV: {e}")

In [ ]:
df.tail(5)

In [ ]:
#JSON Local
ruta_json = f"{ruta_base}add/json/alumnos_fuente_001.json"
try:
  df_json = pd.read_json(ruta_json)
  print_profile(df_json, "df_json")
except Exception as e:
  print(f"Error al leer el archivo JSON: {e}")
  estado_global = "error"

In [ ]:
! head -n 20 {ruta_json}

In [ ]:
#PARQUET Local
ruta_parquet = f"{ruta_base}add/parquet"
try:
  df_json = pd.read_json(ruta_json)
  print_profile(df_json, "df_json")
except Exception as e:
  print(f"Error al leer el archivo JSON: {e}")
  estado_global = "error"

In [ ]:
#AVRO Local
ruta_avro = f"{ruta_base}add/avro/"
registros = []
try:
  archivos_avro = [
      f for f in os.listdir(ruta_avro) if f.endswith(".avro")
  ]
  print("Numero de Archivos", len(archivos_avro))
  print(archivos_avro)
  for a in archivos_avro:
    with open(f"{ruta_avro}{a}", "rb") as f:
      avro_reader = fastavro.reader(f)
      print(avro_reader.writer_schema)
      for record in avro_reader:
        registros.append(record)

    print(f"Registros encontrados: {len(registros)}")
    print(registros)
    df_avro = pd.DataFrame(registros)
    print_profile(df_avro, "df_avro")

except Exception as e:
  print(f"Error al leer el archivo AVRO: {e}")
  estado_global = "error"

In [ ]:
#Lectura http
#file://
#hdfs://
#gs://
#s3://
#ftp://
print(ruta_csv)
import requests

# sobrecarga un archivo
def download_file(url:str, out_path:str, timeout=60):
  r = requests.get(url, timeout=timeout) #200: todo bien en la petecion, #400: error capa 8, #500: error de servidor
  print("Codigo de respuesta", r.status_code)
  r.raise_for_status()
  os.makedirs(os.path.dirname(out_path), exist_ok=True)
  with open(out_path, "wb") as f:
    f.write(r.content)

In [ ]:
# url = "archivo_csv"
out_file = f"{ruta_base}add/descarga/csv/equipos_fuente_001.csv"
out_file = "csv/equipos_fuente_001.csv";
try:
  download_file(url, out_file)
  print("Archivo descargado correctamente")
except Exception as e:
  print("Error al descargar archivo:", e)

In [ ]:
url_parquet = "URL"
out_parquet = f"{ruta_base}add/descarga/opentarget/disease.parquet"
# https://www.ebi.ac.uk/gwas/api/search/downloads/studies

try:
  download_file(url_parquet, out_parquet)
  print("Archivo descargado correctamente")
  dfp = pd.read_parquet(out_parquet)
  print_profile(dfp, "dfp")
except Exception as e:
  print("Error al descargar archivo:", e)

In [ ]:
url_tsv = "URL"
out_tsv = f"{ruta_base}add/descarga/tsv/catalogo.tsv"

# archivos separados por algo
try:
  download_file(url_tsv, out_tsv)
  print("Archivo descargado correctamente")
  dft = pd.read_csv(out_tsv, sep='\t')
  print_profile(dft, "dft")
except Exception as e:
  print("Error al descargar archivo:", e)

In [ ]:
#separado por lo que quieras, siempre y cuando tenga lógica
dfp = pd.read_csv(out_tsv, sep='\t')
print_profile(dfp, "dfp")

In [ ]:
! pwd

/content
